###### Partie 1 – Configuration

In [1]:
%%writefile src/collecte.py
"""Collecte des disponibilités Vélib' et envoi vers BigQuery."""
import logging
from pathlib import Path

import pandas as pd
import pandas_gbq
import requests

PROJET = "velocite-velib"
DATASET = "velib"
URL_INFO = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
URL_STATUS = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

Overwriting src/collecte.py


## Partie 2 – Traces d'exécution (logs)

In [2]:
%%writefile -a src/collecte.py


DOSSIER_LOGS = Path(__file__).resolve().parent.parent / "logs"
DOSSIER_LOGS.mkdir(exist_ok=True)
logging.basicConfig(
    filename=DOSSIER_LOGS / "collecte.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8",
)

Appending to src/collecte.py


## Partie 3 – Appel de l'API

In [3]:
%%writefile -a src/collecte.py


def appeler_api(url):
    """Appelle l'API et renvoie les stations sous forme de DataFrame."""
    reponse = requests.get(url, timeout=30)
    reponse.raise_for_status()
    return pd.DataFrame(reponse.json()["data"]["stations"])

Appending to src/collecte.py


In [4]:
%%writefile -a src/collecte.py


def appeler_api(url):
    """Appelle l'API, vérifie le code HTTP et renvoie les stations en DataFrame."""
    reponse = requests.get(url, timeout=30)
    nom_flux = url.split("/")[-1]
    logging.info(f"Appel API {nom_flux} : code HTTP {reponse.status_code}")
    if reponse.status_code != 200:
        raise ValueError(f"Réponse inattendue de l'API ({nom_flux}) : code {reponse.status_code}")
    return pd.DataFrame(reponse.json()["data"]["stations"])

Appending to src/collecte.py


In [5]:
!python src/collecte.py

In [6]:
print(open("logs/collecte.log", encoding="utf-8").read())

2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-

## Partie 4 – Table stations

In [7]:
%%writefile -a src/collecte.py


def preparer_stations(df_info):
    """Table stations : infos fixes de chaque station."""
    df_info["accepte_cb"] = df_info["rental_methods"].notna()
    colonnes = ["station_id", "stationCode", "name", "lat", "lon", "capacity", "accepte_cb"]
    return df_info[colonnes].drop_duplicates(subset="station_id")

Appending to src/collecte.py


## Partie 5 – Table disponibilites

In [8]:
%%writefile -a src/collecte.py


def preparer_disponibilites(df_status, df_stations):
    """Table disponibilites : une photo de chaque station par heure."""
    df = df_status.copy()
    df["nb_mecaniques"] = df["num_bikes_available_types"].str[0].str["mechanical"]
    df["nb_electriques"] = df["num_bikes_available_types"].str[1].str["ebike"]

    df["last_reported"] = pd.to_datetime(df["last_reported"], unit="s", utc=True)
    df["date_collecte"] = pd.Timestamp.now(tz="UTC")
    # Clé unique : station + heure de collecte (ex. 213688169_2026101309)
    df["cle"] = df["station_id"].astype(str) + "_" + df["date_collecte"].dt.strftime("%Y%m%d%H")

    df = df.merge(df_stations[["station_id", "capacity"]], on="station_id", how="left")
    df = df[(df["capacity"] > 0) & (df["is_installed"] == 1)]
    df["taux_remplissage"] = df["num_bikes_available"] / df["capacity"]
    df["station_vide"] = df["num_bikes_available"] == 0
    df["station_pleine"] = df["num_docks_available"] == 0

    colonnes = ["cle", "station_id", "num_bikes_available", "num_docks_available",
                "nb_mecaniques", "nb_electriques", "is_renting", "is_returning",
                "last_reported", "date_collecte", "taux_remplissage",
                "station_vide", "station_pleine"]
    return df[colonnes].drop_duplicates(subset="cle")

Appending to src/collecte.py


## Partie 6 – Anti-doublons

In [9]:
%%writefile -a src/collecte.py


def retirer_deja_presentes(df):
    """Anti-doublons : on retire les lignes dont la clé est déjà dans BigQuery."""
    requete = f"SELECT cle FROM `{PROJET}.{DATASET}.disponibilites`"
    try:
        deja = pandas_gbq.read_gbq(requete, project_id=PROJET)
    except Exception:
        logging.info("Table disponibilites absente : première collecte")
        return df
    return df[~df["cle"].isin(deja["cle"])]

Appending to src/collecte.py


## Partie 7 – Journal des exécutions

In [10]:
%%writefile -a src/collecte.py


def ecrire_journal(statut, nb_lignes, message):
    """Trace de chaque exécution dans la table journal_executions."""
    journal = pd.DataFrame([{
        "date_execution": pd.Timestamp.now(tz="UTC"),
        "statut": statut,
        "nb_lignes": nb_lignes,
        "message": message,
    }])
    pandas_gbq.to_gbq(journal, f"{DATASET}.journal_executions", project_id=PROJET, if_exists="append")

Appending to src/collecte.py


## Partie 8 – Programme principal

In [11]:
%%writefile -a src/collecte.py


def main():
    logging.info("Début de la collecte")
    try:
        df_stations = preparer_stations(appeler_api(URL_INFO))
        df_dispo = preparer_disponibilites(appeler_api(URL_STATUS), df_stations)
        nb_recues = len(df_dispo)
        df_dispo = retirer_deja_presentes(df_dispo)

        pandas_gbq.to_gbq(df_stations, f"{DATASET}.stations", project_id=PROJET, if_exists="replace")
        if len(df_dispo) > 0:
            pandas_gbq.to_gbq(df_dispo, f"{DATASET}.disponibilites", project_id=PROJET, if_exists="append")

        message = f"{nb_recues} lignes reçues, {len(df_dispo)} nouvelles envoyées"
        logging.info(message)
        ecrire_journal("OK", len(df_dispo), message)
    except Exception as erreur:
        logging.error(f"Échec de la collecte : {erreur}")
        try:
            ecrire_journal("ERREUR", 0, str(erreur))
        except Exception:
            pass
        raise


if __name__ == "__main__":
    main()

Appending to src/collecte.py


## Test du script

In [12]:
!python src/collecte.py


Downloading:   0%|          |
Downloading: 100%|##########|



100%|##########| 1/1 [00:00<00:00, 2192.53it/s]

100%|##########| 1/1 [00:00<00:00, 5562.74it/s]


In [13]:
print(open("logs/collecte.log", encoding="utf-8").read())

2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-

In [14]:
%%writefile lancer_collecte.bat
@echo off
cd /d "C:\Users\utilisateur\velib suz"
call "C:\Users\utilisateur\anaconda3\Scripts\activate.bat"
python src\collecte.py

Overwriting lancer_collecte.bat


In [15]:
!schtasks /create /tn "Collecte Velib" /tr "\"C:\Users\utilisateur\velib suz\lancer_collecte.bat\"" /sc hourly /f

Opération réussie : la tâche planifiée "Collecte Velib" a été créée.


In [16]:
!schtasks /run /tn "Collecte Velib"

Information : la tâche planifiée "Collecte Velib" est en cours d’exécution.
Opération réussie : tentative d’exécution de la tâche planifiée "Collecte Velib".


In [17]:
print(open("logs/collecte.log", encoding="utf-8").read())


2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-

In [41]:
!python src/collecte.py


Downloading:   0%|          |
Downloading: 100%|##########|



100%|##########| 1/1 [00:00<00:00, 4755.45it/s]

100%|##########| 1/1 [00:00<00:00, 2445.66it/s]


## Requêtes SQL

In [18]:
import os
os.makedirs("sql", exist_ok=True)

### Contrôle qualité : doublons

In [19]:
%%writefile sql/00_controle_doublons.sql
-- Contrôle qualité : vérifie qu'il n'y a aucun doublon
-- Si nb_lignes = nb_cles_uniques, chaque photo (station + heure) est unique
SELECT
  COUNT(*) AS nb_lignes,
  COUNT(DISTINCT cle) AS nb_cles_uniques
FROM `velocite-velib.velib.disponibilites`

Overwriting sql/00_controle_doublons.sql


### Requête 1 – Stations vides maintenant

In [20]:
%%writefile sql/01_stations_vides_maintenant.sql
-- Requête 1 : stations vides lors de la dernière collecte, les plus grandes d'abord
SELECT s.name, s.stationCode, s.capacity, d.date_collecte
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
WHERE d.station_vide
  AND d.date_collecte = (SELECT MAX(date_collecte) FROM `velocite-velib.velib.disponibilites`)
ORDER BY s.capacity DESC

Overwriting sql/01_stations_vides_maintenant.sql


### Requête 2 – Situation par heure

In [21]:
%%writefile sql/02_situation_par_heure.sql
-- Requête 2 : situation moyenne par heure (heure de Paris)
SELECT
  EXTRACT(HOUR FROM DATETIME(date_collecte, "Europe/Paris")) AS heure,
  ROUND(100 * AVG(taux_remplissage), 1) AS taux_remplissage_moyen_pct,
  ROUND(100 * COUNTIF(station_vide) / COUNT(*), 1) AS pct_stations_vides,
  ROUND(100 * COUNTIF(station_pleine) / COUNT(*), 1) AS pct_stations_pleines
FROM `velocite-velib.velib.disponibilites`
GROUP BY heure
ORDER BY heure

Overwriting sql/02_situation_par_heure.sql


### Requête 3 – Top 10 des stations le plus souvent vides

In [22]:
%%writefile sql/03_top10_stations_vides.sql
-- Requête 3 : top 10 des stations le plus souvent vides
SELECT
  s.station_id, s.name, s.stationCode,
  COUNT(*) AS nb_photos,
  COUNTIF(d.station_vide) AS nb_fois_vide,
  ROUND(100 * COUNTIF(d.station_vide) / COUNT(*), 1) AS pct_temps_vide
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
GROUP BY s.station_id, s.name, s.stationCode
ORDER BY pct_temps_vide DESC, nb_fois_vide DESC
LIMIT 10

Overwriting sql/03_top10_stations_vides.sql


### Requête 4 – KPI du réseau
État du réseau lors de la dernière collecte : vélos disponibles (mécaniques / électriques) et places hors service.

In [23]:
%%writefile sql/04_kpi_reseau.sql
-- Requête 4 : état du réseau lors de la dernière collecte
SELECT
  SUM(d.num_bikes_available) AS velos_disponibles,
  SUM(d.nb_mecaniques) AS velos_mecaniques,
  SUM(d.nb_electriques) AS velos_electriques,
  SUM(GREATEST(s.capacity - d.num_bikes_available - d.num_docks_available, 0)) AS places_hors_service
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
WHERE d.date_collecte = (SELECT MAX(date_collecte) FROM `velocite-velib.velib.disponibilites`)

Overwriting sql/04_kpi_reseau.sql


### Requête 5 – Secteurs les plus demandés
Départs estimés par zone : on compare chaque photo à la précédente (fonction `LAG`). Une baisse du nombre de vélos = des départs. C'est une estimation minimale (flux nets par heure).

In [24]:
%%writefile sql/05_secteurs_plus_demandes.sql
-- Requête 5 : secteurs les plus demandés (départs nets estimés entre deux photos)
WITH variations AS (
  SELECT
    station_id,
    num_bikes_available
      - LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY date_collecte) AS variation
  FROM `velocite-velib.velib.disponibilites`
),
zones AS (
  SELECT
    station_id,
    IF(SAFE_CAST(stationCode AS INT64) < 21000,
       CONCAT("Paris ", CAST(DIV(SAFE_CAST(stationCode AS INT64), 1000) AS STRING)),
       "Hors Paris") AS zone
  FROM `velocite-velib.velib.stations`
)
SELECT
  z.zone,
  SUM(IF(v.variation < 0, -v.variation, 0)) AS departs_estimes,
  SUM(IF(v.variation > 0, v.variation, 0)) AS arrivees_estimees
FROM variations AS v
JOIN zones AS z ON v.station_id = z.station_id
GROUP BY z.zone
ORDER BY departs_estimes DESC
LIMIT 10%%writefile sql/05_secteurs_plus_demandes.sql
-- Requête 5 : secteurs les plus demandés (départs nets estimés entre deux photos)
WITH variations AS (
  SELECT
    station_id,
    num_bikes_available
      - LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY date_collecte) AS variation
  FROM `velocite-velib.velib.disponibilites`
),
zones AS (
  SELECT
    station_id,
    IF(SAFE_CAST(stationCode AS INT64) < 21000,
       CONCAT("Paris ", CAST(DIV(SAFE_CAST(stationCode AS INT64), 1000) AS STRING)),
       "Hors Paris") AS zone
  FROM `velocite-velib.velib.stations`
)
SELECT
  z.zone,
  SUM(IF(v.variation < 0, -v.variation, 0)) AS departs_estimes,
  SUM(IF(v.variation > 0, v.variation, 0)) AS arrivees_estimees
FROM variations AS v
JOIN zones AS z ON v.station_id = z.station_id
GROUP BY z.zone
ORDER BY departs_estimes DESC
LIMIT 10%%writefile sql/05_secteurs_plus_demandes.sql
-- Requête 5 : secteurs les plus demandés (départs nets estimés entre deux photos)
WITH variations AS (
  SELECT
    station_id,
    num_bikes_available
      - LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY date_collecte) AS variation
  FROM `velocite-velib.velib.disponibilites`
),
zones AS (
  SELECT
    station_id,
    IF(SAFE_CAST(stationCode AS INT64) < 21000,
       CONCAT("Paris ", CAST(DIV(SAFE_CAST(stationCode AS INT64), 1000) AS STRING)),
       "Hors Paris") AS zone
  FROM `velocite-velib.velib.stations`
)
SELECT
  z.zone,
  SUM(IF(v.variation < 0, -v.variation, 0)) AS departs_estimes,
  SUM(IF(v.variation > 0, v.variation, 0)) AS arrivees_estimees
FROM variations AS v
JOIN zones AS z ON v.station_id = z.station_id
GROUP BY z.zone
ORDER BY departs_estimes DESC
LIMIT 10

Overwriting sql/05_secteurs_plus_demandes.sql


### Vue pour le dashboard
Vue BigQuery qui joint les deux tables et prépare les champs utiles au dashboard (heure de Paris, zone, indicateurs 0/1, coordonnées GPS).

In [25]:
%%writefile sql/06_vue_dashboard.sql
-- Vue utilisée par le dashboard Data Studio
CREATE OR REPLACE VIEW `velocite-velib.velib.vue_dashboard` AS
SELECT
  d.*,
  s.name, s.stationCode, s.capacity, s.accepte_cb,
  DATETIME(d.date_collecte, "Europe/Paris") AS date_collecte_paris,
  EXTRACT(HOUR FROM DATETIME(d.date_collecte, "Europe/Paris")) AS heure,
  IF(SAFE_CAST(s.stationCode AS INT64) < 21000,
     CONCAT("Paris ", CAST(DIV(SAFE_CAST(s.stationCode AS INT64), 1000) AS STRING)),
     "Hors Paris") AS zone,
  GREATEST(s.capacity - d.num_bikes_available - d.num_docks_available, 0) AS places_hors_service,
  CAST(d.station_vide AS INT64) AS est_vide,
  CAST(d.station_pleine AS INT64) AS est_pleine,
  CONCAT(CAST(s.lat AS STRING), ",", CAST(s.lon AS STRING)) AS coordonnees
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id

Overwriting sql/06_vue_dashboard.sql


## Publication sur GitHub

In [26]:
!git init
!git branch -M main

Reinitialized existing Git repository in C:/Users/utilisateur/velib suz/.git/


In [27]:
!git config user.name "suzy5670"
!git config user.email "mdane230@gmail.com"

In [28]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/collecte.py
	modified:   velib2.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


## Préparation de la publication sur GitHub
Avant d'envoyer le projet, on définit ce qui ne doit **jamais** être publié et les bibliothèques nécessaires pour que le projet puisse être repris par une autre personne.

### Fichier `.gitignore`
Liste des fichiers ignorés par Git : les secrets (`*.json`, `.env`), les logs et les fichiers temporaires de Jupyter.

In [29]:
%%writefile .gitignore
# Secrets : ne jamais publier
*.json
.env

# Fichiers générés automatiquement
logs/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


### Fichier `requirements.txt`
Bibliothèques Python du projet, avec leurs versions. Installation : `pip install -r requirements.txt`.

In [30]:
%%writefile requirements.txt
requests==2.34.2
pandas==2.3.3
pandas-gbq==0.35.2

Overwriting requirements.txt


### Nettoyage
Suppression d'un dossier vide `collecte.py` créé par erreur à la racine (le vrai script est dans `src/`).

In [31]:
import shutil, os

if os.path.isdir("collecte.py"):
    shutil.rmtree("collecte.py")
    print("Faux dossier supprimé ✅")
else:
    print("Rien à supprimer")

Rien à supprimer


### Vérification avant envoi
On vérifie qu'aucun fichier sensible (logs, clés) ne fait partie des fichiers à publier.

In [32]:
!git status




On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/collecte.py
	modified:   velib2.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


## Publication sur GitHub
Envoi du projet sur le dépôt public https://github.com/suzy5670/velocite-velib

### Préparer l'envoi (commit)
`git add` sélectionne les fichiers, `git commit` enregistre une version du projet avec un message.

In [33]:
!git add .
!git commit -m "Collecte Velib vers BigQuery : script, requetes SQL, notebook"

[main 98883ce] Collecte Velib vers BigQuery : script, requetes SQL, notebook
 2 files changed, 297 insertions(+), 520 deletions(-)


In [34]:
!git remote add origin https://github.com/suzy5670/velocite-velib.git
!git push -u origin main

error: remote origin already exists.


branch 'main' set up to track 'origin/main'.


To https://github.com/suzy5670/velocite-velib.git
   6084a76..98883ce  main -> main


## Automatisation avec GitHub Actions
Le workflow `.github/workflows/collecte.yml` lance la collecte toutes les heures sur les serveurs de GitHub (même PC éteint). Connexion à BigQuery sans clé, grâce à la fédération d'identité (Workload Identity Federation).

In [35]:
import os
os.makedirs(".github/workflows", exist_ok=True)

In [36]:
%%writefile .github/workflows/collecte.yml
name: Collecte Velib

on:
  schedule:
    - cron: "5 * * * *"   # toutes les heures, à HH:05 (heure UTC)
  workflow_dispatch:       # bouton pour lancer à la main

permissions:
  contents: read
  id-token: write          # autorise GitHub à prouver son identité à Google

jobs:
  collecte:
    runs-on: ubuntu-latest
    steps:
      - name: Récupérer le code
        uses: actions/checkout@v4

      - name: Installer Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Installer les bibliothèques
        run: pip install -r requirements.txt

      - name: Connexion à Google Cloud (sans clé)
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: projects/50689790778/locations/global/workloadIdentityPools/github/providers/github-actions
          service_account: velib-collecte-15@velocite-velib.iam.gserviceaccount.com

      - name: Lancer la collecte
        run: python src/collecte.py

      - name: Afficher les logs
        if: always()
        run: cat logs/collecte.log

Overwriting .github/workflows/collecte.yml


In [37]:
!git add .
!git commit -m "Ajout de l'automatisation GitHub Actions"
!git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Everything up-to-date


In [38]:
!git add .
!git commit -m "Ajout des requetes 04, 05 et de la vue du dashboard"
!git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Everything up-to-date


In [42]:
!python src/collecte.py


Downloading:   0%|          |
Downloading: 100%|##########|



100%|##########| 1/1 [00:00<00:00, 3563.55it/s]

100%|##########| 1/1 [00:00<00:00, 1350.39it/s]


In [43]:
print(open("logs/collecte.log", encoding="utf-8").read())

2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-